# Model B - Lightweight CNN (EuroSAT)

EN3150 Assignment 03. Model B will use depthwise-separable convolutions with at most **100,000 trainable parameters**.

This first stage establishes the shared data pipeline only. Architecture, optimizer experiments, training and evaluation will be added in subsequent commits.

## Data contract

Run `Dataset.ipynb` once before this notebook, from the repository root. We reuse `data/eurosat_split.csv` and `data/eurosat/train|val|test/<class>/` without downloading, modifying or re-splitting images.

- 27,000 native 64x64 RGB images; 10 alphabetically ordered classes.
- Stratified 70/15/15 split: 18,900 training, 4,050 validation, 4,050 test images.
- Pixels scaled from 0-255 to float32 in 0-1; no augmentation.
- Training order changes each epoch. Validation and test order remain fixed.
- Test files are checked for structural consistency here; no test predictions or model selection occur.


In [ ]:
from pathlib import Path
import hashlib
import platform

import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = Path("data/eurosat")
SPLIT_CSV = Path("data/eurosat_split.csv")
IMG_SIZE = (64, 64)
SEED = 42
BATCH_SIZE = 128  # Match Model A's final training batch size.
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_COUNTS = {"train": 18_900, "val": 4_050, "test": 4_050}

tf.keras.utils.set_random_seed(SEED)
print("Python:", platform.python_version(), "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none")


## 1. Verify the committed split and local files

Fail early if local folders differ from the shared manifest. Image identities are `(label, filename)` pairs. Checks cover duplicate identities, allowed labels/splits, stratification, and exact file membership. The manifest hash identifies the split used by future experiments.


In [ ]:
assert SPLIT_CSV.is_file() and DATA_DIR.is_dir(), (
    "Shared data is missing. Run Dataset.ipynb from the repository root first."
)
manifest = pd.read_csv(SPLIT_CSV, dtype=str)
assert list(manifest.columns) == ["file", "label", "split"], "Unexpected manifest columns"
assert not manifest.isna().any().any(), "Manifest contains missing values"
assert not manifest.duplicated(["label", "file"]).any(), "An image appears more than once in the manifest"
assert set(manifest["label"]) == set(CLASS_NAMES), "Unexpected classes"
assert set(manifest["split"]) == set(EXPECTED_COUNTS), "Unexpected split names"
assert manifest["file"].map(lambda name: Path(name).name == name and name.lower().endswith(".jpg")).all(), "Invalid image filenames"
assert len(manifest) == 27_000, "Expected 27,000 images"

counts = pd.crosstab(manifest["label"], manifest["split"]).reindex(
    index=CLASS_NAMES, columns=list(EXPECTED_COUNTS)
)
assert not counts.isna().any().any(), "A class is missing from a split"
np.testing.assert_allclose(
    counts.to_numpy() / counts.sum(axis=1).to_numpy()[:, None],
    np.tile([0.70, 0.15, 0.15], (NUM_CLASSES, 1)), rtol=0, atol=1e-12,
)

expected_paths = {
    f"{row.split}/{row.label}/{row.file}" for row in manifest.itertuples(index=False)
}
actual_paths = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()}
missing = sorted(expected_paths - actual_paths)
extra = sorted(actual_paths - expected_paths)
assert not missing and not extra, f"Dataset differs from manifest: missing={missing[:5]}, extra={extra[:5]}"
for split, expected in EXPECTED_COUNTS.items():
    assert int(counts[split].sum()) == expected, f"Wrong count for {split}"

SPLIT_SHA256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print(counts.to_string())
print("Totals:", counts.sum().to_dict())
print("Split SHA-256:", SPLIT_SHA256)
print("Manifest and local file membership agree.")


## 2. Load and normalise images

Read JPEGs directly instead of resizing them: an unexpected size or channel count must fail rather than be silently corrected. Validate decoded files in all splits once, without caching their pixels in RAM. Each float32 image occupies about 48 KiB, so caching all 27,000 images would require about 1.24 GiB before pipeline overhead.

Shuffle file paths before decoding, then batch and prefetch. Recreating the training dataset with the same seed gives each future optimizer experiment the same initial shuffle state. Validation and test paths are sorted and never shuffled.


In [ ]:
def decode_image(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=0)
    image = tf.ensure_shape(image, IMG_SIZE + (3,))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


split_paths = {}
split_labels = {}
for split in EXPECTED_COUNTS:
    rows = manifest.loc[manifest["split"] == split].sort_values(["label", "file"])
    split_paths[split] = [str(DATA_DIR / row.split / row.label / row.file) for row in rows.itertuples(index=False)]
    split_labels[split] = np.array([CLASS_NAMES.index(label) for label in rows["label"]], dtype=np.int32)


def make_dataset(split, batch_size=BATCH_SIZE):
    """Create a fresh deterministic pipeline; only training is shuffled each epoch."""
    if split not in EXPECTED_COUNTS:
        raise ValueError(f"Unknown split: {split}")
    ds = tf.data.Dataset.from_tensor_slices((split_paths[split], split_labels[split]))
    if split == "train":
        ds = ds.shuffle(EXPECTED_COUNTS[split], seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


print("Class mapping:", dict(enumerate(CLASS_NAMES)))
# Consume every image once to surface malformed JPEGs and wrong shapes now.
# Validation is structural only; no model or test-performance metric is involved.
for split in EXPECTED_COUNTS:
    seen = 0
    for images, labels in make_dataset(split):
        tf.debugging.assert_type(images, tf.float32)
        tf.debugging.assert_all_finite(images, "Non-finite pixels")
        tf.debugging.assert_greater_equal(images, 0.0)
        tf.debugging.assert_less_equal(images, 1.0)
        tf.debugging.assert_greater_equal(labels, 0)
        tf.debugging.assert_less(labels, NUM_CLASSES)
        seen += int(tf.shape(images)[0])
    assert seen == EXPECTED_COUNTS[split], f"Wrong decoded count for {split}"
    print(f"{split}: {seen:,} decoded 64x64 RGB images validated")

# Fresh pipelines after validation; the training shuffle has not been consumed.
train_ds = make_dataset("train")
val_ds = make_dataset("val")
test_ds = make_dataset("test")


## Next stage

Add `build_model_b()`, verify its output probabilities, and explicitly assert the **trainable** parameter budget. No architecture, training run, test evaluation or result exports are performed in this stage.
